In [10]:
import sys
sys.path.append("..")

import numpy as np

from src.data_loader import KITTILoader

from src.proxies.gps_proxy import extract_all_gps_proxies
from src.proxies.imu_proxy import extract_all_imu_proxies
from src.proxies.camera_proxy import extract_all_camera_proxies
from src.proxies.lidar_proxy import extract_all_lidar_proxies

In [11]:
datasets = KITTILoader.load_multiple_drives()

print(
    f"Loaded {len(datasets)} drives"
)

Loading drive 0009...
Loading drive 0015...
Loading drive 0051...
Loading drive 0091...

Loaded 4 drives.
Loaded 4 drives


In [12]:
all_gps = {}
all_imu = {}

for drive, loader in datasets.items():

    print(f"\nProcessing {drive}")

    data = loader.raw_data

    dt = loader.get_dt()

    gps = extract_all_gps_proxies(
        data.oxts,
        dt=dt
    )

    imu = extract_all_imu_proxies(
        data.oxts,
        dt=dt
    )

    all_gps[drive] = gps
    all_imu[drive] = imu

    print(
        f"Frames: {len(gps['speed'])}"
    )


Processing 0009
Frames: 447

Processing 0015
Frames: 297

Processing 0051
Frames: 438

Processing 0091
Frames: 340


In [13]:
all_cam = {}

for drive, loader in datasets.items():

    data = loader.raw_data

    frames = []

    limit = min(
        50,
        len(data.cam2_files)
    )

    for i in range(limit):

        frames.append(
            np.array(
                data.get_cam2(i)
            )
        )

    cam = extract_all_camera_proxies(
        frames
    )

    all_cam[drive] = cam

    print(
        drive,
        len(cam["flow_magnitude"])
    )

0009 49
0015 49
0051 49
0091 49


In [14]:
all_lidar = {}

for drive, loader in datasets.items():

    data = loader.raw_data

    scans = [
        data.get_velo(i)
        for i in range(
            min(
                50,
                len(data.velo_files)
            )
        )
    ]

    oxts = data.oxts[:50]

    lidar = extract_all_lidar_proxies(
        scans,
        oxts
    )

    all_lidar[drive] = lidar

    print(
        drive,
        np.mean(
            lidar["icp_residual"]
        )
    )

Processing scan 20/50
Processing scan 40/50
0009 0.20854406619400742
Processing scan 20/50
Processing scan 40/50
0015 0.25376864242927666
Processing scan 20/50
Processing scan 40/50
0051 0.27044089615172473
Processing scan 20/50
Processing scan 40/50
0091 0.20069765573764084
